# Large Language Models

## Introduction

Large language models (LLMs) are deep neural networks trained on massive text corpora to predict the next token in a sequence. Despite this simple objective, they develop remarkable capabilities: they can answer questions, write code, summarize documents, translate languages, and reason through complex problems.

In this lecture, we will cover:

* The Transformer architecture and self-attention mechanism
* Tokenization and autoregressive text generation
* The training pipeline: pretraining, fine-tuning, and RLHF
* Prompting techniques: zero-shot, few-shot, and chain-of-thought
* Evaluation methods and their limitations

You already know feedforward networks, backpropagation, and PyTorch from the deep learning lecture. You learned policy gradients and reward-based optimization in the RL lecture. LLMs combine both: a deep neural network trained first by maximum likelihood on text, then refined with reinforcement learning from human feedback.

![AI confidently lies to you (caniphish.com)](https://caniphish.com/Supporting/Memes/AI/ai-liar-meme.jpg)

*LLMs can produce fluent, confident text that is completely wrong, a phenomenon called "hallucination." Understanding why this happens (and how to mitigate it) requires understanding how these models work under the hood, which is what this lecture is about.*

## The Transformer Architecture

In the deep learning lecture, we briefly mentioned the Transformer as the architecture behind modern LLMs. Now let's understand how it works.

### From recurrence to attention

Before Transformers, the dominant approach for processing sequences (text, time series) was **recurrent neural networks (RNNs)**. An RNN processes tokens one at a time, maintaining a hidden state $h_t$ that summarizes everything seen so far:

$$h_t = f(W_h h_{t-1} + W_x x_t + b)$$

This creates two problems:

1. **Sequential bottleneck**: You cannot compute $h_t$ until you have $h_{t-1}$. This prevents parallelization and makes training slow.
2. **Long-range dependencies**: Information from early tokens must survive through many sequential updates. In practice, the hidden state "forgets" early tokens, even with gated architectures like LSTMs.

The key insight of **attention** is: instead of compressing the entire sequence into a fixed-size hidden state, let each position directly "look at" all other positions and decide what is relevant.

### Self-attention

Self-attention is the core operation of the Transformer. Given a sequence of $n$ token embeddings $X \in \mathbb{R}^{n \times d}$, we compute three matrices:

$$Q = XW_Q, \quad K = XW_K, \quad V = XW_V$$

where $W_Q, W_K, W_V \in \mathbb{R}^{d \times d_k}$ are learned weight matrices. $Q$ (queries), $K$ (keys), and $V$ (values) are analogous to a lookup system: each query asks "what should I attend to?", keys provide "what do I contain?", and the dot product $QK^T$ measures how relevant each key is to each query.

The attention output is:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

The $\sqrt{d_k}$ scaling prevents dot products from growing too large as the dimension increases, which would push the softmax into saturated regions with vanishing gradients (recall the sigmoid saturation problem from the deep learning lecture).

Let's compute self-attention from scratch:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Suppose we have a sequence of 4 tokens, each with embedding dimension 8
tokens = ["The", "patient", "has", "diabetes"]
n_tokens = len(tokens)
d_model = 8
d_k = 4  # dimension of queries/keys/values

# Random token embeddings (in practice, these are learned)
X = np.random.randn(n_tokens, d_model)

# Learned projection matrices
W_Q = np.random.randn(d_model, d_k) * 0.5
W_K = np.random.randn(d_model, d_k) * 0.5
W_V = np.random.randn(d_model, d_k) * 0.5

# Compute Q, K, V
Q = X @ W_Q  # (4, 4)
K = X @ W_K  # (4, 4)
V = X @ W_V  # (4, 4)

# Compute attention scores
scores = Q @ K.T / np.sqrt(d_k)  # (4, 4)
print("Raw attention scores:")
print(np.round(scores, 2))

# Apply softmax row-wise
def softmax(x, axis=-1):
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

attention_weights = softmax(scores)
print("\nAttention weights (each row sums to 1):")
print(np.round(attention_weights, 3))

# Compute output
output = attention_weights @ V  # (4, 4)

# Visualize attention weights as a heatmap
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(attention_weights, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(n_tokens))
ax.set_yticks(range(n_tokens))
ax.set_xticklabels(tokens)
ax.set_yticklabels(tokens)
ax.xaxis.set_label_position("top")
ax.xaxis.tick_top()
ax.set_xlabel("Key (attending to)")
ax.set_ylabel("Query (attending from)")
ax.set_title("Self-Attention Weights", pad=30)
for i in range(n_tokens):
    for j in range(n_tokens):
        ax.text(j, i, f"{attention_weights[i, j]:.2f}",
                ha="center", va="center", fontsize=10,
                color="white" if attention_weights[i, j] > 0.5 else "black")
plt.colorbar(im)
plt.tight_layout()
plt.show()

Each row of the attention weight matrix shows how much each token attends to every other token. For example, the word "diabetes" might attend strongly to "patient" because they are semantically related.

### Causal masking

In LLMs (which generate text left to right), each token should only attend to tokens that came before it, not future tokens. This is enforced with a **causal mask**: we set the upper-triangular entries of the score matrix to $-\infty$ before applying softmax, so their attention weights become zero.

In [ ]:
# Causal mask: token i can only attend to tokens 0, 1, ..., i
mask = np.triu(np.ones((n_tokens, n_tokens)), k=1) * (-1e9)
masked_scores = scores + mask
causal_weights = softmax(masked_scores)

print("Causal attention weights:")
print(np.round(causal_weights, 3))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(attention_weights, cmap="Blues", vmin=0, vmax=1)
axes[0].set_xticks(range(n_tokens))
axes[0].set_yticks(range(n_tokens))
axes[0].set_xticklabels(tokens)
axes[0].set_yticklabels(tokens)
axes[0].xaxis.tick_top()
axes[0].set_title("Bidirectional (BERT-style)", pad=30)

axes[1].imshow(causal_weights, cmap="Blues", vmin=0, vmax=1)
axes[1].set_xticks(range(n_tokens))
axes[1].set_yticks(range(n_tokens))
axes[1].set_xticklabels(tokens)
axes[1].set_yticklabels(tokens)
axes[1].xaxis.tick_top()
axes[1].set_title("Causal / autoregressive (GPT-style)", pad=30)
for i in range(n_tokens):
    for j in range(n_tokens):
        axes[1].text(j, i, f"{causal_weights[i, j]:.2f}",
                     ha="center", va="center", fontsize=10,
                     color="white" if causal_weights[i, j] > 0.5 else "black")

plt.tight_layout()
plt.show()

Notice that in causal attention, the first token ("The") can only attend to itself, while "diabetes" can attend to all four tokens.

### Multi-head attention

A single attention head can only capture one type of relationship at a time. **Multi-head attention** runs $h$ attention heads in parallel, each with its own $W_Q^{(i)}, W_K^{(i)}, W_V^{(i)}$ matrices, and concatenates their outputs:

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W_O$$

where $\text{head}_i = \text{Attention}(XW_Q^{(i)}, XW_K^{(i)}, XW_V^{(i)})$.

Different heads learn to attend to different things. For example, one head might capture syntactic relationships (subject-verb agreement), another might capture semantic relationships (disease-symptom), and another might attend to nearby positions.

### Positional encoding

Self-attention is **permutation-equivariant**: if you shuffle the input tokens, the outputs shuffle in the same way (no position information is used). More informally, attention is "order-agnostic" since it does not know which token came first. But word order matters in language! "The patient treated the doctor" means something very different from "The doctor treated the patient."

The Transformer injects positional information by adding **positional encodings** to the token embeddings. The original Transformer uses sinusoidal functions:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right), \quad PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)$$

where $pos$ is the position in the sequence and $i$ is the dimension index. Each position gets a unique pattern of sines and cosines, allowing the model to distinguish positions.

In [ ]:
def positional_encoding(max_len, d_model):
    pe = np.zeros((max_len, d_model))
    position = np.arange(max_len)[:, np.newaxis]
    div_term = 10000 ** (2 * np.arange(d_model // 2) / d_model)
    pe[:, 0::2] = np.sin(position / div_term)
    pe[:, 1::2] = np.cos(position / div_term)
    return pe

pe = positional_encoding(max_len=64, d_model=32)

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(pe, cmap="RdBu", aspect="auto")
ax.set_xlabel("Embedding dimension")
ax.set_ylabel("Position in sequence")
ax.set_title("Sinusoidal Positional Encoding")
plt.colorbar(im)
plt.tight_layout()
plt.show()

Each row is a different position's encoding vector. High-frequency dimensions (left side, small $i$) oscillate rapidly with position, while low-frequency dimensions (right side, large $i$) change slowly. This gives the model both fine-grained and coarse position information.

### The full Transformer block

A Transformer block combines self-attention with a feedforward network and residual connections:

1. **Multi-head self-attention** with residual connection and layer normalization
2. **Feedforward network** (two linear layers with a nonlinearity) with residual connection and layer normalization

$$\begin{aligned}
h' &= \text{LayerNorm}(x + \text{MultiHeadAttention}(x)) \\
\text{output} &= \text{LayerNorm}(h' + \text{FFN}(h'))
\end{aligned}$$

The residual connections (adding $x$ back to the output) help with training deep networks by allowing gradients to flow directly through the addition, bypassing the attention and FFN layers. This is similar to the skip connections in ResNets.

![Full Transformer architecture (Jay Alammar)](https://jalammar.github.io/images/t/transformer_resideual_layer_norm_3.png)

*The full Transformer architecture from Jay Alammar's "The Illustrated Transformer." Left: two stacked encoder blocks, each containing Self-Attention and Feed Forward layers with Add & Normalize (residual + layer norm) after each. Right: two stacked decoder blocks with an additional Encoder-Decoder Attention layer. Positional encodings are added to the input embeddings at the bottom.*

The animated GIF below shows how a trained Transformer generates output tokens one at a time. At each step, the decoder attends to all encoder outputs and all previously generated tokens, then predicts the next token.

![Transformer decoding animation (Jay Alammar)](https://jalammar.github.io/images/t/transformer_decoding_2.gif)

*Autoregressive decoding in the Transformer: the decoder generates one token per step, feeding each output back as input for the next step.*

Modern LLMs stack many such blocks. GPT-2 has 12-48 blocks, GPT-3 has 96, and GPT-4 is rumored to have even more. Each block refines the representations, building increasingly abstract and useful features.

### Decoder-only vs. encoder-only vs. encoder-decoder

There are three main Transformer variants:

| Architecture | Attention | Use Case | Examples |
|---|---|---|---|
| **Encoder-only** | Bidirectional (see all tokens) | Understanding, classification | BERT, RoBERTa |
| **Decoder-only** | Causal (see only past tokens) | Text generation | GPT, LLaMA, Claude |
| **Encoder-decoder** | Encoder bidirectional, decoder causal + cross-attention | Seq-to-seq (translation, summarization) | T5, BART |

Modern LLMs (GPT-4, Claude, LLaMA, Gemini) are overwhelmingly **decoder-only**. The reason is simple: a decoder-only model can do everything. It generates text autoregressively and can be prompted to perform classification, translation, summarization, and more, all with the same architecture.

### Question

In the self-attention computation, the attention weight matrix $A = \text{softmax}(QK^T / \sqrt{d_k})$ has shape $(n, n)$ where $n$ is the sequence length.

1. What is the computational complexity of computing $QK^T$ as a function of sequence length $n$ and key dimension $d_k$?
2. If you double the sequence length from 2,048 to 4,096 tokens, by what factor does the memory needed to store the attention matrix increase?
3. Why does this explain the existence of "context window" limits in LLMs?

### Answer

1. Computing $QK^T$ is a matrix multiplication of $(n, d_k) \times (d_k, n)$, which requires $O(n^2 d_k)$ operations.
2. The attention matrix has $n^2$ entries. Doubling $n$ from 2,048 to 4,096 increases the matrix size by a factor of $4$ (from ~4M to ~16M entries).
3. Context windows are limited because both compute and memory scale **quadratically** with sequence length. A model processing 100K tokens needs 10,000x the attention memory of a model processing 1K tokens. This is why extending context windows is an active research area, with approaches like sparse attention, sliding window attention, and FlashAttention that reduce the effective cost.

## Tokenization and Text Generation

### How text becomes numbers

Neural networks operate on numbers, not characters. **Tokenization** is the process of converting text into a sequence of integer IDs that the model can process.

There are three approaches to tokenization:

**Character-level**: Each character is a token. Simple, but sequences become very long (a 1,000-word document becomes ~5,000 characters), and individual characters carry little meaning.

**Word-level**: Each word is a token. Intuitive, but the vocabulary must be enormous to handle all possible words, including misspellings, technical jargon, and morphological variants ("running", "runs", "ran").

**Subword (BPE)**: The sweet spot. Byte Pair Encoding (BPE) starts with individual characters and iteratively merges the most frequent adjacent pair into a new token. Common words like "the" become single tokens, while rare words get split into meaningful subwords: "biostatistics" might become ["bio", "stat", "istics"].

*Predictive text on phones uses a similar idea: the model learns which token sequences are common and predicts the next one. LLMs take this to a much larger scale.*

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Tokenize a simple sentence
text = "Biostatistics is the application of statistics to biological problems."
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.encode(text)

print(f"Text: {text}")
print(f"Tokens: {tokens}")
print(f"Token IDs: {token_ids}")
print(f"Number of tokens: {len(tokens)}")
print(f"Vocabulary size: {tokenizer.vocab_size}")

In [ ]:
# Medical text gets split into more subwords
medical_terms = [
    "electroencephalography",
    "diabetes",
    "hypertension",
    "the",
    "patient",
    "CRISPR-Cas9",
]

print("Token breakdown for medical terms:")
print("-" * 50)
for term in medical_terms:
    toks = tokenizer.tokenize(term)
    print(f"{term:30s} -> {toks} ({len(toks)} tokens)")

Common words like "the" and "patient" are single tokens, while specialized terms like "electroencephalography" get split into multiple subwords. This is why LLMs process technical text more slowly (more tokens) and why token counts matter for context window limits.

### Autoregressive generation

LLMs generate text one token at a time, left to right. At each step:

1. Feed all previous tokens into the model
2. The model outputs a probability distribution over the entire vocabulary for the next token
3. Sample a token from this distribution
4. Append it to the sequence and repeat

This is called **autoregressive** generation because each prediction depends on all previous predictions.

In [ ]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("gpt2")
model.eval()

prompt = "The patient was diagnosed with"
input_ids = tokenizer.encode(prompt, return_tensors="pt")

# Get next-token probabilities
with torch.no_grad():
    outputs = model(input_ids)
    logits = outputs.logits[0, -1, :]  # logits for the last position
    probs = torch.softmax(logits, dim=0)

# Top 10 most likely next tokens
top_k = 10
top_probs, top_indices = torch.topk(probs, top_k)
top_tokens = [tokenizer.decode(idx) for idx in top_indices]

print(f"Prompt: '{prompt}'")
print(f"\nTop {top_k} next token predictions:")
for token, prob in zip(top_tokens, top_probs):
    print(f"  '{token}': {prob:.4f}")

In [ ]:
# Visualize the effect of temperature on sampling
temperatures = [0.5, 1.0, 2.0]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, temp in zip(axes, temperatures):
    scaled_logits = logits / temp
    scaled_probs = torch.softmax(scaled_logits, dim=0)
    top_p, top_i = torch.topk(scaled_probs, top_k)
    top_t = [tokenizer.decode(idx) for idx in top_i]

    ax.barh(range(top_k), top_p.numpy(), color="steelblue")
    ax.set_yticks(range(top_k))
    ax.set_yticklabels(top_t)
    ax.set_xlabel("Probability")
    ax.set_title(f"Temperature = {temp}")
    ax.invert_yaxis()

plt.suptitle(f"Next token after: '{prompt}'", fontsize=12)
plt.tight_layout()
plt.show()

**Temperature** controls the "sharpness" of the distribution:

* $T < 1$: Sharper distribution, more deterministic. The model becomes more confident in its top choice.
* $T = 1$: Original distribution as computed by the model.
* $T > 1$: Flatter distribution, more random. The model spreads probability mass more evenly.

At $T = 0$ (greedy decoding), the model always picks the most likely token. This produces repetitive, boring text. At high temperature, the model produces creative but potentially incoherent text. In practice, values around $T = 0.7$-$1.0$ work well for most tasks.

**Top-k sampling** restricts sampling to the $k$ most likely tokens (e.g., $k = 50$). Tokens outside the top $k$ are excluded, which prevents the model from picking very unlikely tokens. The downside is that $k$ is fixed regardless of how confident the model is.

**Top-p (nucleus) sampling** addresses this: instead of a fixed $k$, only consider the smallest set of tokens whose cumulative probability exceeds $p$ (e.g., $p = 0.9$). When the model is confident (one token has 90% probability), top-p selects just that token. When the model is uncertain (many tokens with similar probability), top-p includes more candidates. This adapts the number of candidate tokens to the model's confidence at each step.

### Question

Consider a BPE tokenizer that produces the following splits:

```
"unhappiness" -> ['un', 'happ', 'iness']   (3 tokens)
"happiness"   -> ['happ', 'iness']          (2 tokens)
```

1. Why does the tokenizer split "unhappiness" into 3 tokens but "happiness" into 2? How does BPE decide where to split?
2. If a model has a context window of 4,096 tokens, approximately how many English words can fit? (The average ratio is roughly 1.3 tokens per word for English.)
3. Why might this ratio be higher for non-English languages or specialized domains like genomics?

### Answer

1. BPE builds its vocabulary by iteratively merging the most frequent character pairs in the training corpus. "happ" and "iness" appeared together frequently enough to become tokens, but "unhappiness" as a whole did not. The prefix "un" is a common prefix that was merged separately. BPE splits words at the boundaries of the subwords it learned during training.
2. Approximately $4096 / 1.3 \approx 3{,}150$ words.
3. Non-English languages may use characters or scripts not well-represented in the (predominantly English) training data, so each word gets split into more subwords. For genomics, DNA sequences (ATCGATCG...) use a tiny alphabet but produce very long token sequences because the BPE tokenizer was not trained on genomic data and doesn't have efficient subword units for nucleotide patterns.

## The Training Pipeline

Training a modern LLM involves three stages, each building on the previous one. This pipeline is what turns a randomly initialized neural network into a helpful assistant.

### Stage 1: Pretraining

The foundation of every LLM is **pretraining**: training the model to predict the next token on a massive text corpus.

The objective is **causal language modeling** (next-token prediction):

$$\mathcal{L}(\theta) = -\sum_{t=1}^{T} \log P_\theta(x_t \mid x_1, x_2, \ldots, x_{t-1})$$

This is the negative log-likelihood of the training data. Minimizing this loss is equivalent to maximum likelihood estimation (MLE), connecting directly to the statistical framework you have seen throughout this course. The optimization uses Adam (a variant of SGD with adaptive learning rates), the same family of gradient-based optimizers you studied in Module 2.

**Scale of pretraining:**

| Model | Parameters | Training Tokens | Training Cost (est.) |
|-------|-----------|----------------|---------------------|
| GPT-2 (2019) | 1.5B | 40B | ~$50K |
| GPT-3 (2020) | 175B | 300B | ~$5M |
| LLaMA 2 (2023) | 70B | 2T | ~$25M |
| LLaMA 3 (2024) | 405B | 15T | ~$100M+ |

The training data typically includes web crawls (Common Crawl), books, Wikipedia, scientific papers, and code repositories. The model sees billions of sentences across every domain, learning grammar, facts, reasoning patterns, and even code syntax, all from the single objective of predicting the next token.

**Emergent capabilities.** As models get larger and see more data, they develop abilities not explicitly trained for:

* **In-context learning**: Following instructions or examples provided in the prompt, without any parameter updates
* **Chain-of-thought reasoning**: Working through multi-step problems
* **Code generation**: Writing working programs
* **Translation**: Converting between languages

These capabilities "emerge" at scale, meaning they are absent in small models but appear once models cross certain size thresholds. This is one of the most surprising findings in modern AI research.

### Stage 2: Supervised fine-tuning (SFT)

After pretraining, the model is a powerful text completion engine: given "The capital of France is", it will generate "Paris." But it won't follow instructions like "Summarize this clinical trial report in 3 bullet points." It just continues text in whatever direction seems statistically likely.

**Supervised fine-tuning** trains the model on curated (instruction, response) pairs:

```
Instruction: What are the common side effects of metformin?
Response: Common side effects of metformin include nausea, diarrhea,
stomach pain, and decreased appetite. These are usually mild and
improve over time...
```

The loss function is the same as pretraining (next-token prediction), but now applied only to the response tokens, conditioned on the instruction.

**Parameter-efficient fine-tuning (LoRA).** Full fine-tuning updates all model parameters, which is expensive for large models. **Low-Rank Adaptation (LoRA)** freezes the original weights and adds small trainable adapter matrices:

$$W' = W + BA$$

where $W \in \mathbb{R}^{d \times d}$ is the frozen pretrained weight matrix, $B \in \mathbb{R}^{d \times r}$ and $A \in \mathbb{R}^{r \times d}$ are trainable, and $r \ll d$ (typically $r = 8$ or $16$). Instead of updating $d^2$ parameters, LoRA only trains $2dr$ parameters, a reduction by a factor of $d/(2r)$. For a model with $d = 4096$ and $r = 16$, this is a 128x reduction.

### Stage 3: RLHF

Supervised fine-tuning makes the model follow instructions, but it doesn't teach the model *which* responses are better. Should the model be concise or verbose? Formal or casual? How should it handle ambiguous or sensitive questions?

**Reinforcement Learning from Human Feedback (RLHF)** addresses this by optimizing the model's outputs based on human preferences. This is where the RL concepts from the previous lecture directly apply.

RLHF has three steps:

**Step 1: Collect human preferences.** Given a prompt, generate two responses from the model. A human annotator chooses which response is better. Collect thousands of such comparisons.

**Step 2: Train a reward model.** The reward model $R_\phi(x, y)$ learns to score how good a response $y$ is for a given prompt $x$. It is trained on the preference data using the **Bradley-Terry model**:

$$P(y_A \succ y_B \mid x) = \frac{\exp(R_\phi(x, y_A))}{\exp(R_\phi(x, y_A)) + \exp(R_\phi(x, y_B))} = \sigma(R_\phi(x, y_A) - R_\phi(x, y_B))$$

This is logistic regression on the reward difference. The reward model learns to assign higher scores to responses that humans prefer.

**Step 3: Optimize the LLM with PPO.** Using the reward model as the environment's reward signal, optimize the LLM policy $\pi_\theta$ using Proximal Policy Optimization (PPO, a policy gradient method from the RL lecture). The objective includes a KL divergence penalty to prevent the model from drifting too far from the SFT model:

$$\mathcal{J}(\theta) = \mathbb{E}_{x \sim D,\, y \sim \pi_\theta(\cdot|x)}\left[R_\phi(x, y) - \beta \, D_{\text{KL}}(\pi_\theta(\cdot|x) \,\|\, \pi_{\text{SFT}}(\cdot|x))\right]$$

The KL penalty prevents **reward hacking**: without it, the model might find degenerate outputs that score high on the reward model but are actually nonsensical (e.g., repeating certain phrases the reward model was tricked into liking). Note the parallel to variational inference: in VI, the ELBO includes a KL divergence term $D_{\text{KL}}(q \| p)$ that keeps the approximate posterior close to the prior. Here, the KL term keeps the RLHF policy close to the SFT policy, serving a similar regularization role.

```mermaid
flowchart LR
    subgraph Stage 1
        A["<b>Pretraining</b><br/>Next-token prediction<br/>on web-scale text<br/>(trillions of tokens)"]
    end
    subgraph Stage 2
        B["<b>Supervised<br/>Fine-Tuning (SFT)</b><br/>Instruction-response pairs<br/>(~100K examples)"]
    end
    subgraph Stage 3a
        C["<b>Reward Model</b><br/>Human preferences<br/>(A &gt; B comparisons)<br/>Bradley-Terry"]
    end
    subgraph Stage 3b
        D["<b>RLHF (PPO)</b><br/>Optimize policy<br/>against reward model<br/>+ KL penalty"]
    end
    A --> B --> C --> D
```

### Question

In RLHF, we train a reward model from human preference data. Suppose we have 3 responses to the prompt "Explain what a p-value is":

* **Response A**: Correct, detailed, with a worked example (3 paragraphs)
* **Response B**: Correct and concise (2 sentences)
* **Response C**: Confident but contains a subtle error (defines p-value as "the probability that the null hypothesis is true")

A human annotator ranks them A > B > C.

1. Using the Bradley-Terry model, express $P(A \succ C)$ in terms of their reward scores $r_A$ and $r_C$.
2. Why do we add a KL penalty between the RLHF policy and the SFT policy? What failure mode does this prevent?
3. In the RL lecture, we discussed the exploration-exploitation tradeoff. Does RLHF face the same tradeoff? Why or why not?

### Answer

1. $P(A \succ C) = \frac{\exp(r_A)}{\exp(r_A) + \exp(r_C)} = \sigma(r_A - r_C)$, where $\sigma$ is the sigmoid function. This is identical to logistic regression on the reward difference.
2. The KL penalty prevents **reward hacking**. Without it, the model might find outputs that exploit weaknesses in the reward model, receiving high reward scores for degenerate text. For example, the model might learn to produce very long, hedge-heavy responses that the reward model likes but humans would find unhelpful. The KL penalty keeps the RLHF model close to the SFT model, which already produces reasonable text.
3. RLHF does face an exploration-exploitation tradeoff, but it is less severe than in typical RL settings. The policy starts from a well-trained SFT model (not from scratch), so it already produces reasonable responses. The KL penalty also limits how far the model can "explore" away from the SFT policy. In contrast, RL agents in games often start from random policies and must explore extensively to discover good strategies.

## Prompting Techniques

Once an LLM is trained, how you communicate with it (the **prompt**) has a large effect on the quality of the output. This is sometimes called "prompt engineering," though the principles are straightforward.

### Zero-shot prompting

Simply ask the model to perform the task with no examples:

In [ ]:
import anthropic

client = anthropic.Anthropic()

# Zero-shot: classify a clinical note
note = "Patient presents with increased thirst, frequent urination, and fatigue. Fasting glucose 180 mg/dL."

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=100,
    messages=[{
        "role": "user",
        "content": f"Classify this clinical note as 'diabetes-related' or 'not diabetes-related'. Respond with just the classification.\n\nNote: {note}"
    }]
)
print(f"Zero-shot: {response.content[0].text}")

### Few-shot prompting

Provide examples in the prompt to demonstrate the desired input-output format:

In [ ]:
# Few-shot: provide examples first
few_shot_prompt = """Classify clinical notes as 'diabetes-related' or 'not diabetes-related'.

Note: "Patient reports numbness and tingling in feet, HbA1c 8.2%."
Classification: diabetes-related

Note: "Chest X-ray shows bilateral infiltrates consistent with pneumonia."
Classification: not diabetes-related

Note: "Follow-up for Type 2 DM, current medications include metformin 1000mg BID."
Classification: diabetes-related

Note: "{note}"
Classification:"""

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=100,
    messages=[{"role": "user", "content": few_shot_prompt.format(note=note)}]
)
print(f"Few-shot: {response.content[0].text}")

Few-shot prompting works because of **in-context learning**: the model recognizes the pattern in the examples and applies it to the new input, without any parameter updates. This is one of the emergent capabilities of large models.

### Chain-of-thought prompting

For tasks requiring reasoning, ask the model to show its work:

In [ ]:
# Chain-of-thought: ask the model to reason step by step
cot_prompt = f"""Analyze this clinical note and determine if it is diabetes-related.
Think through the evidence step by step before giving your classification.

Note: "{note}"

Step-by-step analysis:"""

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=300,
    messages=[{"role": "user", "content": cot_prompt}]
)
print(f"Chain-of-thought:\n{response.content[0].text}")

Chain-of-thought prompting works because it forces the model to break the problem into intermediate steps, each of which is easier for the model than jumping directly to the answer. This is especially helpful for math, logic, and multi-step reasoning tasks.

### System prompts and structured output

**System prompts** set the model's behavior and persona:

In [ ]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=300,
    system="You are a clinical NLP assistant. Extract structured information from clinical notes. Always respond in valid JSON format.",
    messages=[{
        "role": "user",
        "content": f"Extract the following from this note: symptoms, lab values, and likely diagnosis.\n\nNote: {note}"
    }]
)
print(response.content[0].text)

Structured output (JSON, tables, specific formats) is useful for building pipelines where LLM outputs feed into downstream code. Modern APIs also support **tool use** (function calling), where the model can invoke external functions. We will explore this in detail in the next lecture on AI agents.

### Question

You want an LLM to extract medication names from clinical notes. Consider these two prompting strategies:

**Strategy A** (zero-shot):
```
Extract all medication names from the following clinical note: {note}
```

**Strategy B** (few-shot):
```
Extract medication names from clinical notes.

Note: "Patient takes metformin 500mg and lisinopril 10mg daily."
Medications: metformin, lisinopril

Note: "Started on amoxicillin for infection. Continue atorvastatin."
Medications: amoxicillin, atorvastatin

Note: "No current medications. Advised lifestyle changes."
Medications: none

Note: "{note}"
Medications:
```

1. Why might Strategy B produce more reliable results than Strategy A?
2. Why is the third example (with "none") important?
3. What could go wrong if all few-shot examples contained exactly 2 medications?

### Answer

1. Strategy B shows the model the exact output format (comma-separated names, no dosages, no extra text) and demonstrates that only drug names should be extracted. Strategy A is ambiguous about format: the model might return full sentences, include dosages, or use a different format each time.
2. The "none" example teaches the model what to do when there are no medications. Without it, the model might hallucinate medication names to avoid an empty output, since all its examples showed at least one medication.
3. If all examples contain exactly 2 medications, the model might be biased toward extracting exactly 2, even when the note contains 1, 3, or more. This is why few-shot examples should cover the range of expected outputs (different counts, edge cases, negatives).

## Evaluation

### Perplexity

The most basic evaluation metric for language models is **perplexity**, which measures how well the model predicts text:

$$\text{PPL} = \exp\left(-\frac{1}{T}\sum_{t=1}^{T} \log P(x_t \mid x_{<t})\right)$$

This is the exponential of the average cross-entropy loss. It has an intuitive interpretation: a perplexity of $k$ means the model is, on average, as uncertain as if it were choosing uniformly among $k$ options at each step.

* PPL = 1: The model is perfectly certain about every next token.
* PPL = 10: The model is as uncertain as choosing among 10 equally likely options.
* PPL = 50,000 (vocabulary size): The model is guessing randomly.

Lower perplexity means better prediction.

In [ ]:
# Compute perplexity of different texts using GPT-2
def compute_perplexity(text, model, tokenizer):
    input_ids = tokenizer.encode(text, return_tensors="pt")
    with torch.no_grad():
        outputs = model(input_ids, labels=input_ids)
        loss = outputs.loss  # average cross-entropy over all tokens
    return torch.exp(loss).item()

texts = {
    "Natural English": "The patient was admitted to the hospital for observation after the surgery.",
    "Technical (medical)": "Immunohistochemical staining revealed CD20-positive B-cell lymphoma.",
    "Grammatically wrong": "Patient the was hospital admitted to for surgery the observation.",
    "Random tokens": "Purple seventeen elephant calculus Montreal guitar quantum.",
}

print("Perplexity comparison:")
print("-" * 60)
for label, text in texts.items():
    ppl = compute_perplexity(text, model, tokenizer)
    print(f"{label:30s}: PPL = {ppl:.1f}")

Natural, grammatical English gets low perplexity (the model predicts it well). Shuffled or random text gets high perplexity (the model is surprised by each token).

### Benchmarks

Perplexity measures prediction quality but not whether the model is useful. For that, we use **benchmarks** that test specific capabilities:

| Benchmark | What it tests | Format |
|-----------|-------------|--------|
| **MMLU** | Broad knowledge across 57 subjects | Multiple choice |
| **HumanEval** | Code generation (Python) | Write function from docstring |
| **GSM8K** | Grade-school math reasoning | Word problems |
| **TruthfulQA** | Resistance to common misconceptions | Open-ended questions |
| **GPQA** | Graduate-level science questions | Multiple choice |

### Limitations of benchmarks

Benchmarks have significant limitations:

* **Data contamination**: If benchmark questions appear in the training data, the model is "cheating" by memorizing answers rather than reasoning.
* **Goodhart's law**: "When a measure becomes a target, it ceases to be a good measure." Models can be optimized to score well on benchmarks without being genuinely more capable.
* **Narrow coverage**: Benchmarks test specific skills but miss many real-world capabilities (nuance, creativity, safety, helpfulness).

**LLM-as-judge** is an alternative approach: use one LLM to evaluate the outputs of another. While this introduces its own biases, it scales better than human evaluation and can assess open-ended responses that benchmarks cannot.

### Question

A language model assigns the following per-token log probabilities (base $e$) for the sentence "The cat sat on the mat": $[-2.3, -3.1, -4.0, -1.5, -0.8, -2.5]$.

1. Compute the perplexity of this sentence by hand.
2. If a model achieves a perplexity of 1.0 on a dataset, what does that mean?
3. Why is perplexity alone insufficient for evaluating whether an LLM is helpful?

### Answer

1. Average negative log probability: $(2.3 + 3.1 + 4.0 + 1.5 + 0.8 + 2.5) / 6 = 14.2 / 6 \approx 2.367$. Perplexity $= \exp(2.367) \approx 10.67$. The model is, on average, as uncertain as choosing among ~11 options at each token.
2. Perplexity 1.0 means $\exp(0) = 1$, so the average log probability is 0, meaning $P(x_t | x_{<t}) = 1$ for every token. The model is perfectly certain about every next token. On natural language data, this would indicate memorization or data leakage. (It could also occur on trivially deterministic data, but that is not a realistic evaluation setting.)
3. Perplexity measures how well the model predicts text, not whether its outputs are helpful, safe, truthful, or well-formatted. A model could have low perplexity (good at predicting text) while producing biased, toxic, or factually wrong outputs when prompted. Perplexity also doesn't capture instruction-following ability or the quality of generated responses.

## Scaling Laws and the LLM Landscape

### Scaling laws

An important empirical finding is that LLM performance follows **power-law scaling** with model size, dataset size, and compute:

$$L(N, D) \approx \left(\frac{N_c}{N}\right)^{\alpha_N} + \left(\frac{D_c}{D}\right)^{\alpha_D} + L_\infty$$

where $L$ is the loss, $N$ is the number of parameters, $D$ is the number of training tokens, and $\alpha_N, \alpha_D$ are scaling exponents.

The **Chinchilla scaling law** (Hoffmann et al., 2022) established that for a given compute budget, you should scale model size and data equally. This meant many earlier models (like GPT-3) were **undertrained**: they had many parameters but were trained on too few tokens relative to their size. The rule of thumb is approximately 20 training tokens per parameter.

### Open vs. closed models

| Category | Examples | Access |
|----------|---------|--------|
| **Closed (API only)** | GPT-4, Claude, Gemini | Pay per token, no weight access |
| **Open-weight** | LLaMA 3, Mistral, DeepSeek | Download weights, run locally |
| **Fully open** | OLMo, DBRX | Weights + training data + code |

Open models enable local deployment (privacy, cost), custom fine-tuning, and research. Closed models typically offer better performance and easier integration but with less control. The gap between open and closed models has narrowed significantly since 2023.

### What runs where

* **7B parameters**: Runs on a consumer GPU (24GB VRAM) or even a laptop with quantization
* **70B parameters**: Requires multiple GPUs or a high-end workstation
* **400B+ parameters**: Requires a cluster; typically accessed via API

![xkcd 2169: Predictive Models](https://imgs.xkcd.com/comics/predictive_models.png)

*LLMs are, at their core, predictive models: they predict the next token. The power comes from the scale of data they've seen and the emergent capabilities that arise from that scale.*

## Summary

| Concept | Key Idea |
|---------|----------|
| Transformer | Self-attention processes all tokens in parallel, replacing sequential RNNs |
| Self-attention | $\text{softmax}(QK^T/\sqrt{d_k})V$: each token attends to all others |
| Causal masking | Decoder-only models mask future tokens for autoregressive generation |
| Tokenization (BPE) | Subword tokenization balances vocabulary size and sequence length |
| Temperature | Controls randomness in sampling: low = deterministic, high = creative |
| Pretraining | Next-token prediction on trillions of tokens (= maximum likelihood) |
| SFT | Fine-tune on instruction-response pairs to follow instructions |
| LoRA | Low-rank adapters for parameter-efficient fine-tuning |
| RLHF | Bradley-Terry reward model + PPO to align with human preferences |
| Prompting | Zero-shot, few-shot, chain-of-thought guide model behavior |
| Perplexity | $\exp(\text{avg cross-entropy})$: measures prediction quality |
| Scaling laws | Performance follows power laws with model size and data |

## Recommended Resources

* Vaswani et al. (2017), ["Attention Is All You Need"](https://arxiv.org/abs/1706.03762) -- the original Transformer paper
* Jay Alammar, ["The Illustrated Transformer"](https://jalammar.github.io/illustrated-transformer/) -- excellent visual explanation
* Jay Alammar, ["The Illustrated GPT-2"](https://jalammar.github.io/illustrated-gpt2/) -- how decoder-only Transformers generate text
* Andrej Karpathy, ["Let's build GPT"](https://www.youtube.com/watch?v=kCc8FmEb1nY) -- build a GPT from scratch in a video lecture
* Hoffmann et al. (2022), ["Training Compute-Optimal Large Language Models"](https://arxiv.org/abs/2203.15556) -- the Chinchilla scaling laws paper
* [Hugging Face Transformers documentation](https://huggingface.co/docs/transformers/) -- practical guide to using pretrained models
* [Anthropic API documentation](https://docs.anthropic.com/) -- using Claude via API